# EMPTY CHAIR
a college workshop as a decision tree build from scratch


# LIBRARIES TO IMPORT

In [1]:
import pandas as pd
import math

# DATA ANALYSING 


## 1 | DATA FETCH 
the data fetching as the initial part in the model after the idea and seach for data 

In [2]:
data = pd.read_csv('KaggleV2-May-2016.csv')
display(data.head())

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## 2 | First analysis of the data 
here i will describe data check missing values change attributes to make it more usefull 

In [3]:
print (f'data informtions : {data.info()}') #check the types of the data 
print('-' * 100)
print(f'count of missing or null values : {data.isna().sum()}')
print('-' * 100)
print(data.nunique())

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  str    
 3   ScheduledDay    110527 non-null  str    
 4   AppointmentDay  110527 non-null  str    
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  str    
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  str    
dtypes: float64(1), int64(8), str(5)
memory usage: 11.8 MB
data informtions : None
---------------------------------------------------------------------------------

## Notes : 
- **Patient Id** and **Appointment ID** : useless information for our model.
- **Boolean Values**: most values are boolean.
- **Handicap** : 5 diffrent classes of the case .
- **Schedule Day** and **Appointment Day** : we need to reformulate them in a usefull way 

## 3 | NORMALISE DATA

- to make it easier to employ and treat we remplace the strings into integers soo we evoid typing errors and we standarized the time type of data so we can apply arithmetic function on them because there have same formmat \
- also i ve remplaced the **ScheduleDay** and the **AppointmentDay** with the *new added collumn* called **WaitingDay** so the model find it more clean way 


In [4]:


data['AppointmentDay'] = pd.to_datetime(data['AppointmentDay'])
data['ScheduledDay'] = pd.to_datetime(data['ScheduledDay'])
data['Gender'] = data['Gender'].map({'M': 1, 'F': 0})
data['WaitingDay'] = (data['AppointmentDay'].dt.normalize() - data['ScheduledDay'].dt.normalize()).dt.days
data['No-show'] = data['No-show'].map({'No': 0, 'Yes' : 1})


display(data.head())
      
#34 47 60 

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,WaitingDay
0,2.987250e+13,5642903,0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,0,0,0
1,5.589978e+14,5642503,1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,0,0,0
2,4.262962e+12,5642549,0,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,0,0,0
3,8.679512e+11,5642828,0,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,0,0,0
4,8.841186e+12,5642494,0,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,0,0,0


## DROP USELESS COLUMN 

the model use decision that really will attract the result but some of them are just needs as a database like address and client id here i need the row not the id of client and where he lives . we need the situation of the client and the actions happend it self not whos happened to i guess the client here is just a *SOMEONE* no matter who real he is 

In [5]:
data = data.drop(columns=['AppointmentID','PatientId','ScheduledDay','AppointmentDay','Neighbourhood'])
display(data.head())

,Gender,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,WaitingDay
0,0,62,0,1,0,0,0,0,0,0
1,1,56,0,0,0,0,0,0,0,0
2,0,62,0,0,0,0,0,0,0,0
3,0,8,0,0,0,0,0,0,0,0
4,0,56,0,1,1,0,0,0,0,0


# 4 | CHECK AND FILTER OUTLIERS AND ANORMAL VALUES 

the mean principle of this part is droping row with anormal infos like negative age and because we have a lot of data i have the right of make it more organized and because of this firtring the model will decrease the error marge 

In [6]:
def age_category(i : int):
    if i > 60 :
        return 3
    elif i > 47:
        return 2
    elif i > 34:
        return 1
    else:
        return 0

print(data['WaitingDay'].value_counts())
data = data[(data['WaitingDay'] >= 0) & (data['WaitingDay'] <= 120)]

data = data[(data['Age'] > 19) & (data['Age'] < 90)]
data['Age'] = data['Age'].map(age_category)
print(data.shape)

print(data['WaitingDay'].value_counts())

WaitingDay
0      38563
2       6725
4       5290
1       5213
7       4906
       ...  
139        1
132        1
117        1
146        1
123        1
Name: count, Length: 131, dtype: int64
(79615, 10)
WaitingDay
0      26013
2       5626
4       4340
1       3919
7       3350
       ...  
104        1
82         1
101        1
107        1
117        1
Name: count, Length: 113, dtype: int64


## 5 | CHECK DATAFRAME AFTER DATA CLEANING

In [7]:
display(data.head())

print(data.describe())
print(data['Handcap'].value_counts())
print(data['No-show'].value_counts())
print(data['Age'].value_counts())



,Gender,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,WaitingDay
0,0,3,0,1,0,0,0,0,0,0
1,1,2,0,0,0,0,0,0,0,0
2,0,3,0,0,0,0,0,0,0,0
4,0,2,0,1,1,0,0,0,0,0
5,0,3,0,1,0,0,0,0,0,2


             Gender           Age   Scholarship  Hipertension     Diabetes  \
count  79615.000000  79615.000000  79615.000000  79615.000000  79615.00000   
mean       0.304026      1.473491      0.093563      0.269120      0.09821   
std        0.459996      1.125593      0.291221      0.443505      0.29760   
min        0.000000      0.000000      0.000000      0.000000      0.00000   
25%        0.000000      0.000000      0.000000      0.000000      0.00000   
50%        0.000000      1.000000      0.000000      0.000000      0.00000   
75%        1.000000      2.000000      0.000000      1.000000      0.00000   
max        1.000000      3.000000      1.000000      1.000000      1.00000   

         Alcoholism       Handcap  SMS_received       No-show    WaitingDay  
count  79615.000000  79615.000000  79615.000000  79615.000000  79615.000000  
mean       0.041789      0.025774      0.331056      0.194222     10.262677  
std        0.200107      0.170894      0.470596      0.395603  

## CUTING TABLE INTO TWO TABlES 

*Because ID3 dont work with numbers only work for categories and cart work with numbers in normal way and i will build both*
- i cut the waiting time to months soo i can categorize them to less number 
- and changed the handicap from 0,1,2,3 as categories into boolean handicap or not to make data of handicap collumn more **Balenced** in the dataset

In [8]:
cart = data.copy()
id3 = data.copy()

id3['WaitingDay'] = id3['WaitingDay'] // 30
id3['Handcap'] = (id3['Handcap'] > 0).astype(int)

display(id3.head(1))

print(id3['Gender'].value_counts())
print(id3['Age'].value_counts())
print(id3['Scholarship'].value_counts())
print(id3['Hipertension'].value_counts())
print(id3['Diabetes'].value_counts())
print(id3['Alcoholism'].value_counts())
print(id3['Handcap'].value_counts())
print(id3['SMS_received'].value_counts())
print(id3['No-show'].value_counts())
print(id3['WaitingDay'].value_counts())

,Gender,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,WaitingDay
0,0,3,0,1,0,0,0,0,0,0


Gender
0    55410
1    24205
Name: count, dtype: int64
Age
0    21209
2    20304
3    19301
1    18801
Name: count, dtype: int64
Scholarship
0    72166
1     7449
Name: count, dtype: int64
Hipertension
0    58189
1    21426
Name: count, dtype: int64
Diabetes
0    71796
1     7819
Name: count, dtype: int64
Alcoholism
0    76288
1     3327
Name: count, dtype: int64
Handcap
0    77717
1     1898
Name: count, dtype: int64
SMS_received
0    53258
1    26357
Name: count, dtype: int64
No-show
0    64152
1    15463
Name: count, dtype: int64
WaitingDay
0    71646
1     6175
2     1639
3      155
Name: count, dtype: int64


## MODEL
building the model from scratch using just basic libraries and pythons codes 

# ENTHROPY FUNCTION

In [9]:
def enthropy(one, target):
    enthropies = []
    for col_name, series in one.items():
        print(col_name)
        val = []
        posetive = 0
        negative = 0
        for v in series.unique():
            mask = (series == v)
            check = target[mask]

            posetive = (check == 1).sum()
            negative = (check == 0).sum()

            val.append((posetive, negative, len(check)))

        enthro = 0
        for pos, neg, total in val:
            if pos == 0 or neg == 0:
                continue
            else:
                enthro += (abs((pos / total) * math.log(( pos / total) , 2 ) + ( neg / total) * math.log(( neg / total), 2))) * (total / len(target))

        enthropies.append(enthro)
    return enthropies

test = id3.copy()
enthropy(test.drop(columns='No-show'), data['No-show'])

Gender
Age
Scholarship
Hipertension


Diabetes
Alcoholism
Handcap
SMS_received
WaitingDay


[np.float64(0.710132294220126),
 np.float64(0.7050069651526945),
 np.float64(0.7095282421086941),
 np.float64(0.7093432986583141),
 np.float64(0.7101110836695833),
 np.float64(0.7102157668789574),
 np.float64(0.7101238944187293),
 np.float64(0.6993858005485327),
 np.float64(0.7044038496814669)]